# XGBoost v0 (Baseline) — Student Test Score Prediction
MSE 546 Final Project — Group 3

Initial XGBoost baseline with default parameters. No hyperparameter tuning or early stopping — serves as a starting point for improvement in v1.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

In [ ]:
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')
sample_sub = pd.read_csv('../data/sample_submission.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')
train_df.head()

## 3. Preprocessing

Label encode categorical features.

In [ ]:
categorical_cols = ['gender', 'course', 'internet_access', 'sleep_quality',
                    'study_method', 'facility_rating', 'exam_difficulty']

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
    le.fit(combined)
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))
    label_encoders[col] = le

print('Encoding complete.')

## 4. Prepare Features and Target

In [ ]:
feature_cols = [c for c in train_df.columns if c not in ['id', 'exam_score']]

X = train_df[feature_cols]
y = train_df['exam_score']
X_test = test_df[feature_cols]

print(f'Features: {feature_cols}')

## 5. Train/Validation Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape}, Validation: {X_val.shape}')

## 6. Train XGBoost Model (Default Parameters)

Using XGBoost with all default hyperparameters — no tuning, no early stopping, only 100 estimators.

In [ ]:
model = XGBRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print('Training complete.')

## 7. Evaluate on Validation Set

In [ ]:
y_pred_val = model.predict(X_val)

mae = mean_absolute_error(y_val, y_pred_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
r2 = r2_score(y_val, y_pred_val)

print('=== Validation Results (v0 Baseline) ===')
print(f'MAE:  {mae:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R²:   {r2:.4f}')

## 8. Actual vs Predicted — Scatter Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(y_val, y_pred_val, alpha=0.3, s=10, color='steelblue', label='Predictions')

min_val = min(y_val.min(), y_pred_val.min())
max_val = max(y_val.max(), y_pred_val.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

ax.set_xlabel('Actual Exam Score', fontsize=12)
ax.set_ylabel('Predicted Exam Score', fontsize=12)
ax.set_title(f'Actual vs Predicted — v0 Baseline (R² = {r2:.4f})', fontsize=14)
ax.legend(fontsize=11)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 9. Error Distribution

In [ ]:
errors = y_val.values - y_pred_val

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of errors
axes[0].hist(errors, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Prediction Error (Actual - Predicted)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Error Distribution — v0 Baseline', fontsize=13)
axes[0].text(0.02, 0.95, f'Mean: {errors.mean():.3f}\nStd: {errors.std():.3f}',
             transform=axes[0].transAxes, fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Residuals vs Predicted
axes[1].scatter(y_pred_val, errors, alpha=0.3, s=10, color='steelblue')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Exam Score', fontsize=11)
axes[1].set_ylabel('Residual (Actual - Predicted)', fontsize=11)
axes[1].set_title('Residuals vs Predicted — v0 Baseline', fontsize=13)

plt.tight_layout()
plt.show()

## 10. Feature Importance

In [ ]:
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance.to_string(index=False))

## 11. Generate Submission

In [ ]:
test_predictions = model.predict(X_test)

submission = pd.DataFrame({
    'id': test_df['id'],
    'exam_score': test_predictions
})

import os
os.makedirs('../submission', exist_ok=True)
submission.to_csv('../submission/xgboost_v0_submission.csv', index=False)

print(f'Submission shape: {submission.shape}')
print(f'Saved to ../submission/xgboost_v0_submission.csv')
submission.head()

## 12. Areas for Improvement

This baseline v0 uses XGBoost with default parameters. Potential improvements for v1 include:

- **More estimators** (100 → 1000) with early stopping to prevent overfitting
- **Hyperparameter tuning**: `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`
- **Regularization**: `reg_alpha` (L1) and `reg_lambda` (L2) to reduce overfitting
- **Lower learning rate** with more trees for finer convergence